In [1]:
import pandas as pd

# Simulate a small dataset of customer reviews
data = {
    "review": [
        "This product is absolutely amazing!",
        "Terrible quality, broke after one day.",
        "It's okay, nothing special.",
        "Best purchase I've made this year!!!",
        "Waste of money. Do not buy.",
        "Pretty good for the price.",
    ],
    "label": ["positive", "negative", "neutral", "positive", "negative", "positive"]
}

df = pd.DataFrame(data)
print(df)
print(f"\nShape: {df.shape}")

                                   review     label
0     This product is absolutely amazing!  positive
1  Terrible quality, broke after one day.  negative
2             It's okay, nothing special.   neutral
3    Best purchase I've made this year!!!  positive
4             Waste of money. Do not buy.  negative
5              Pretty good for the price.  positive

Shape: (6, 2)


In [2]:
# Text cleaning
import re

def clean_text(text):
    text = text.lower()                        # lowercase
    text = re.sub(r'[!?.]+', '.', text)        # normalise punctuation
    text = re.sub(r'\s+', ' ', text).strip()   # remove extra whitespace
    return text

df['clean'] = df['review'].apply(clean_text)
print(df[['review', 'clean']])

                                   review  \
0     This product is absolutely amazing!   
1  Terrible quality, broke after one day.   
2             It's okay, nothing special.   
3    Best purchase I've made this year!!!   
4             Waste of money. Do not buy.   
5              Pretty good for the price.   

                                    clean  
0     this product is absolutely amazing.  
1  terrible quality, broke after one day.  
2             it's okay, nothing special.  
3      best purchase i've made this year.  
4             waste of money. do not buy.  
5              pretty good for the price.  


In [5]:
# Add token count per review
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

df['token_count'] = df['clean'].apply(lambda x: len(tokenizer.tokenize(x)))
df['word_count'] = df['clean'].apply(lambda x: len(x.split()))
df['token_word_ratio'] = (df['token_count'] / df['word_count']).round(2)

print(df[['clean', 'word_count', 'token_count', 'token_word_ratio']])
print(f"\nAverage token/word ratio: {df['token_word_ratio'].mean():.2f}")

                                    clean  word_count  token_count  \
0     this product is absolutely amazing.           5            6   
1  terrible quality, broke after one day.           6            8   
2             it's okay, nothing special.           4            8   
3      best purchase i've made this year.           6            9   
4             waste of money. do not buy.           6            8   
5              pretty good for the price.           5            6   

   token_word_ratio  
0              1.20  
1              1.33  
2              2.00  
3              1.50  
4              1.33  
5              1.20  

Average token/word ratio: 1.43


In [6]:
for word in ["it's", "okay", "nothing", "special"]:
    tokens = tokenizer.tokenize(word)
    print(f"{word:12} → {tokens}")

it's         → ['it', "'", 's']
okay         → ['okay']
nothing      → ['nothing']
special      → ['special']


In [7]:
# Analyse by label
summary = df.groupby('label').agg(
    count=('review', 'count'),
    avg_tokens=('token_count', 'mean'),
    avg_ratio=('token_word_ratio', 'mean')
).round(2)

print(summary)
print(f"\nLabel distribution:\n{df['label'].value_counts()}")

          count  avg_tokens  avg_ratio
label                                 
negative      2         8.0       1.33
neutral       1         8.0       2.00
positive      3         7.0       1.30

Label distribution:
label
positive    3
negative    2
neutral     1
Name: count, dtype: int64
